In [8]:
import ioh
from Practical_Problems_Suite import get_practical_problem
import numpy as np
from scipy.optimize._differentialevolution import differential_evolution
import os

In [9]:
triggers = [ioh.logger.trigger.OnImprovement(), ioh.logger.trigger.Each(10)]

logger = ioh.logger.Analyzer(
    triggers=triggers,
    root=os.path.curdir,
    algorithm_name="RandomSearch",
    folder_name="Results-RS",
    algorithm_info="A simple random search optimizer",
    store_positions=False
)

In [10]:
# Set the dimensionality
dimension:int = 26

# Try to get the Tersoff_potentialC1
problem_instance = get_practical_problem(1126,dim=dimension)
print(problem_instance.meta_data.name)

Spacecraft_Trajectory_OptimizationC1


In [11]:
# Test attaching the ioh logger
problem_instance.attach_logger(logger)

In [12]:
class RandomSearchOptimizer:
    def __init__(self, problem:callable, n_iterations=1000, lower_bound=-5, upper_bound=5):

        if not callable(problem):
            raise ValueError("The problem must be a callable function.")
        
        self.problem = problem
        self.n_iterations = n_iterations
        self.lower_bound = lower_bound
        self.upper_bound = upper_bound
        self.best_solution = None
        self.best_objective = float('inf')

    def optimize(self, **kwargs):

        random_seed = kwargs.get('random_seed', 43)

        if isinstance(random_seed, int):
            rng = np.random.default_rng(random_seed)
        else:
            raise ValueError("random_seed must be an integer.")

        for ii in range(self.n_iterations):
            candidate = rng.uniform(self.lower_bound, self.upper_bound, self.problem.meta_data.n_variables)
            objective_value = self.problem(candidate)
            if objective_value < self.best_objective:
                self.best_objective = objective_value
                self.best_solution = candidate
                print(f"New best solution at iteration {ii}: objective {self.best_objective}")
        
        self.problem.reset()
        return self.best_solution, self.best_objective
    
    def reset(self):
        self.best_solution = None
        self.best_objective = float('inf')

In [18]:
class DifferentialEvolutionOptimizer:
    def __init__(self, problem:callable, n_iterations=1000, lower_bound=-5, upper_bound=5):

        if not callable(problem):
            raise ValueError("The problem must be a callable function.")
        
        self.problem = problem
        self.n_iterations = n_iterations
        self.lower_bound = lower_bound
        self.upper_bound = upper_bound
        self.best_solution = None
        self.best_objective = float('inf')

    def optimize(self, **kwargs):

        popsize = kwargs.get('popsize', 3)

        maxiter = max(self.n_iterations // (popsize*self.dimensions) , 1)



        result = differential_evolution(self.problem, 
                                        bounds=[(self.lower_bound, self.upper_bound)] * self.problem.meta_data.n_variables,
                                        maxiter=maxiter,
                                        popsize=popsize,
                                        seed=kwargs.get('random_seed', 43),
                                        disp=True, polish=False)
        
        self.best_solution = result.x
        self.best_objective = result.fun
        print(f"Best solution found: objective {self.best_objective}")
        
        self.problem.reset()
        return self.best_solution, self.best_objective
    
    def reset(self):
        self.best_solution = None
        self.best_objective = float('inf')
    
    @property
    def dimensions(self):
        return self.problem.meta_data.n_variables

In [ ]:
optimizer = RandomSearchOptimizer(problem_instance, n_iterations=1000, lower_bound=-5, upper_bound=5)
# Run a loop of optimizations
n_runs = 5

for run in range(n_runs):
    print(f"Run {run+1} out of {n_runs}")
    best_solution, best_objective = optimizer.optimize(random_seed=run+50)
    print(f"Best solution: {best_solution}, Best objective: {best_objective}")
    optimizer.reset()

# Detach the logger
problem_instance.detach_logger()

Run 1 out of 5
differential_evolution step 1: f(x)= 40.31887771007107
differential_evolution step 2: f(x)= 40.31887771007107
differential_evolution step 3: f(x)= 40.31887771007107
differential_evolution step 4: f(x)= 40.31887771007107
differential_evolution step 5: f(x)= 40.31887771007107
differential_evolution step 6: f(x)= 40.31887771007107
differential_evolution step 7: f(x)= 40.31887771007107
differential_evolution step 8: f(x)= 40.31887771007107
differential_evolution step 9: f(x)= 40.31887771007107
differential_evolution step 10: f(x)= 40.31887771007107
differential_evolution step 11: f(x)= 40.31887771007107
differential_evolution step 12: f(x)= 40.31887771007107
Best solution found: objective 40.31887771007107
Best solution: [-2.39030488  0.05208599 -4.36063823 -1.39345057  0.57586985 -1.4773903
 -4.39654435  2.06580233  1.0156508  -2.27061641  2.43498729  2.17787343
 -1.25112094 -1.57388221 -2.75580192 -3.53902256  2.8892283   3.92353767
 -0.24157963  1.0001527  -3.72872265 -4.